In [ ]:
import os

def is_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if is_colab():
    !git clone https://github.com/sambeets/EPE2316-Power-System-Planning.git
    !pip install pypsa
    os.chdir("EPE2316-Power-System-Planning/Assignments/DigiLab Assignment")
else:
    print("Running locally, assuming the correct directory is already set.")

# DigiLab Assignment — EPE2316 Power System Planning
**Author:** Duy Tran  
**University of South-Eastern Norway**  
**2026**

---

This assignment covers the main computational topics of EPE2316 Power System Planning:

- **Part 1** — Power flow analysis of a 3-bus system using PyPSA and manual Newton-Raphson
- **Part 2** — Unit Commitment, Economic Dispatch, and Security-Constrained Optimal Power Flow on a larger 6-bus system

Complete all tasks in order. Each task builds on the previous results.

---
# Part 1: Power Flow — 3-Bus System

## The Assignment System

The figure below shows the 3-bus power system you will study in Part 1. All variables marked with **?** are to be determined by solving the power flow equations.

![3-bus power system](01.3-bus_power_system_diagram-1.png)

**System base values:**
- $S_{base} = 100$ MVA
- $V_{base} = 22$ kV
- $Z_{base} = V_{base}^2 / S_{base} = 4.84\ \Omega$

**Bus data:**
| Bus | Type | V [pu] | $\delta$ [rad] | P [MW] | Q [Mvar] |
|-----|------|--------|----------------|--------|----------|
| 1 | Slack | 1.0 | 0.0 | ? | ? |
| 2 | PQ | ? | ? | −1.5 | −0.5 |
| 3 | PV | 1.02 | ? | +2.0 | ? |

**Line data (actual values):**
| Line | From | To | R [Ω] | X [Ω] |
|------|------|----|-------|-------|
| L12 | Bus 1 | Bus 2 | 3 | 8 |
| L13 | Bus 1 | Bus 3 | 15 | 30 |
| L23 | Bus 2 | Bus 3 | 1 | 4 |

## Task 1: Create and Simulate with PyPSA

### Task 1.1 — Build the network

Use PyPSA to create a `Network` object containing all buses, lines, generators, and loads as described above.

> **Tip:** When setting the voltage setpoint of 1.02 pu at Bus 3, use the keyword argument `v_mag_pu_set=1.02` when calling `network.add("Bus", ...)` for Bus 3.

In [1]:
import pypsa
import numpy as np
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
from DigiLab_supplementary_file import *

# Insert your code here


In [ ]:
# Check your score for Task 1.1
score_task_1(network)

### Task 1.2 — Run the power flow

Run a power flow simulation. Print the following results **for all buses**:
- Injected active power [MW]
- Injected reactive power [Mvar]
- Voltage magnitude [pu]
- Voltage angle [degrees]

Expected results:

| | Bus 1 | Bus 2 | Bus 3 |
|--|-------|-------|-------|
| P [MW] | −0.487 | −1.5 | 2.0 |
| Q [Mvar] | −0.625 | −0.5 | 1.167 |
| V [pu] | 1.000 | 1.0086 | 1.02 |
| δ [deg] | 0.000 | 0.0277 | 0.7002 |

In [ ]:
# Run power flow and print results here


In [ ]:
# Check your score for Task 1.2
score_task_2(network)

---
## Task 2: Manual Power Flow — Newton-Raphson

### Task 2.1 — Define the power flow error function

Implement a function that takes the unknown variable vector $y$ and returns the power flow mismatch vector $[\Delta P,\ \Delta Q]$.

For this 3-bus system with Bus 1 as slack, Bus 2 as PQ, and Bus 3 as PV, the unknowns are:

$$y = [\delta_2,\ \delta_3,\ V_2]$$

The mismatch equations are:
$$\Delta P_2 = P_2^{calc} - P_2^{spec}$$
$$\Delta P_3 = P_3^{calc} - P_3^{spec}$$
$$\Delta Q_2 = Q_2^{calc} - Q_2^{spec}$$

> **Note:** The $Y_{bus}$ matrix is provided in `DigiLab_supplementary_file.py`.

In [ ]:
from scipy.optimize import root
from DigiLab_supplementary_file import Y_bus

# Insert your code here


### Task 2.2 — Solve the power flow equations

Use `scipy.optimize.root` to solve the power flow mismatch equations and obtain voltages and angles for all buses. Then calculate the injected active and reactive power at all buses.

Print the same four quantities as in Task 1.2. The solution must match the PyPSA result exactly.

In [ ]:
# Solve and print results here


In [ ]:
# Check your score for Task 2
score_task_3(sol)

---
## Task 3: Voltage Sensitivity Study

Evaluate the reactive power injection from the generator at Bus 3 for different voltage setpoints on $V_3$.

Use either PyPSA or your manual solver. Compute $Q_3$ for the following setpoints:
$$V_3 \in [0.95,\ 1.0,\ 1.05]\ \text{pu}$$

Store the results in the list `Q3_vals` (in Mvar).

In [ ]:
V3_vals = [0.95, 1.0, 1.05]
Q3_vals = []

# Insert your code here


In [ ]:
# Check your score for Task 3
score_task_4(Q3_vals)

---
### End of Part 1
---

# Part 2: Unit Commitment, Economic Dispatch, and OPF — 6-Bus System

## The Assignment System

The figure below shows the 6-bus power system you will study in Part 2.

![6-bus power system](02.6-bus_power_system_diagram-1.png)

**System base values:**
- $S_{base} = 100$ MVA
- $V_{base} = 132$ kV

#### Generator data:
| Name | P set [MW] | P nom [MW] | p min [pu] | V set [pu] | Marginal cost [€/MWh] | Ramp rate [pu] | Start-up cost [€] |
|------|-----------|-----------|-----------|-----------|----------------------|---------------|-------------------|
| G1 | Slack | 200 | 0.01 | 1.02 | 50 | 1.0 | 500 |
| G2 | 300 | 300 | 0.8 | 1.02 | 20 | 0.3 | 500 |
| G3 | 100 | 100 | 0.6 | 1.02 | 30 | 0.5 | 500 |

#### Line data:
| Name | Length [km] | r [Ω/km] | x [Ω/km] | I max [A] | S nom [MVA] |
|------|------------|---------|---------|---------|------------|
| L12 | 5 | 0.1 | 0.4 | 1000 | 228.6 |
| L23 | 5 | 0.1 | 0.4 | 1000 | 228.6 |
| L34 | 10 | 0.2 | 0.4 | 700 | 160.0 |
| L45 | 30 | 0.3 | 0.4 | 500 | 114.3 |
| L56 | 5 | 0.1 | 0.4 | 1000 | 228.6 |
| L61 | 5 | 0.1 | 0.4 | 1000 | 228.6 |

In [ ]:
import pypsa
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
from scipy.optimize import root, minimize, NonlinearConstraint
from utils import check_task_1_1, check_task_1_2, check_task_2_1, check_task_2_2

V_base = 132  # kV
S_base = 100  # MVA
I_base = S_base * 1e6 / (np.sqrt(3) * V_base * 1e3)

load_data = pd.read_csv("load_data.csv", index_col="Hour")
P_load_1_timeseries = load_data["P L1 [MW]"].values
P_load_2_timeseries = load_data["P L2 [MW]"].values
P_load_3_timeseries = load_data["P L3 [MW]"].values
Q_load_1_timeseries = load_data["Q L1 [Mvar]"].values
Q_load_2_timeseries = load_data["Q L2 [Mvar]"].values
Q_load_3_timeseries = load_data["Q L3 [Mvar]"].values

network = pypsa.Network(snapshots=range(len(P_load_1_timeseries)))

---
## Task 4: Build and Simulate the 6-Bus Network

### Task 4.1 — Create the network

Use PyPSA to create the 6-bus network with all buses, lines, generators, and time-varying loads from the tables above.

In [ ]:
# Create the 6-bus network here


In [ ]:
check_task_1_1(network)

### Task 4.2 — Run the power flow time series

Run a power flow over the full load time series. Produce three plots:
1. Voltage magnitudes at all buses over time [pu]
2. Active power injection at all buses over time [MW]
3. Reactive power injection at all buses over time [Mvar]

In [ ]:
# Run the power flow here


In [ ]:
# Plot voltage magnitudes


In [ ]:
# Plot active power injections


In [ ]:
# Plot reactive power injections


In [ ]:
check_task_1_2(network)

---
## Task 5: Unit Commitment

Use PyPSA to solve the **Unit Commitment (UC)** problem for the given time series.

Produce a plot showing:
- Total active load of the system [MW]
- Total spinning capacity of committed units [MW]
- Minimum possible dispatch given the committed units [MW]

> **Hint:** To enable unit commitment in PyPSA, set `committable=True` and define `ramp_limit_up`, `ramp_limit_down`, `p_min_pu`, and `start_up_cost` when adding generators.

**Correct plots will give 12 points to the total score.**

In [ ]:
# Run Unit Commitment optimization here


In [ ]:
# Compute and plot the required quantities here


---
## Task 6: Security-Constrained Optimal Power Flow (OPF)

### Task 6.1 — Define the objective function and constraints

You are given a working power flow solver. Your task is to define the **objective function** (minimize total active power losses) and the **constraint function** for the following constraints:

**Voltage constraints (all buses):**
$$V_{min} \le V_i \le V_{max}, \quad V_{min}=0.95,\ V_{max}=1.05$$

**Reactive power constraints (generator buses):**
$$Q_{min} \le Q_i \le Q_{max}, \quad Q_{min} = -0.3 P_{nom},\ Q_{max} = 0.4 P_{nom}$$

**Line current constraints (all lines):**
$$0 \le |I^{ij}| \le I^{ij}_{max}$$

**Decision variables:**
$$u = [V_1,\ V_3,\ V_5,\ P_3,\ P_5]$$

**Constraint vector order (hint):**
$$cons = [V_2,\ V_4,\ V_6,\ P_1,\ Q_1,\ Q_3,\ Q_5,\ I_{12},\ I_{23},\ I_{34},\ I_{45},\ I_{56},\ I_{61}]$$

In [ ]:
from numeric_data import Y_bus, Y_lines

# Initial operating point
u0 = np.array([1.02, 1.02, 1.02, 300, 100])
th_vals = np.array([0, 0, 0, 0, 0, 0], dtype=float)
V_vals  = np.array([1.02, 1.0, 1.02, 1.0, 1.02, 1.0])
P_vals  = np.array([0.0, -100, 300, -120, 100, -50]) / S_base
Q_vals  = np.array([0.0,  -50, 0.0, -60,  0.0, -25]) / S_base

#### Helper functions (provided)

In [ ]:
def get_power_flow(th_vec, V_vec):
    V_complex = V_vec * (np.cos(th_vec) + 1j * np.sin(th_vec))
    I_inj = Y_bus @ V_complex
    S_inj = V_complex * I_inj.conj()
    return np.real(S_inj), np.imag(S_inj)

def power_flow_error(y, th_vals, V_vals, P_vals, Q_vals):
    th2, th3, th4, th5, th6, V2, V4, V6 = y
    th_vec = th_vals.copy()
    V_vec  = V_vals.copy()
    th_vec[1:] = [th2, th3, th4, th5, th6]
    V_vec[1] = V2; V_vec[3] = V4; V_vec[5] = V6
    P_res, Q_res = get_power_flow(th_vec, V_vec)
    return np.array([
        P_res[1]-P_vals[1], P_res[2]-P_vals[2], P_res[3]-P_vals[3],
        P_res[4]-P_vals[4], P_res[5]-P_vals[5],
        Q_res[1]-Q_vals[1], Q_res[3]-Q_vals[3], Q_res[5]-Q_vals[5]
    ])

def get_pf_sol(th_init, V_init, P_init, Q_init):
    """Returns all powers in pu."""
    th_vals = th_init.copy(); V_vals = V_init.copy()
    P_vals  = P_init.copy();  Q_vals = Q_init.copy()
    x0 = np.array([0, 0, 0, 0, 0, 1, 1, 1], dtype=float)
    pf_sol = root(power_flow_error, x0=x0, args=(th_vals, V_vals, P_vals, Q_vals))
    th_vals[1:] = pf_sol.x[:5]
    V_vals[1] = pf_sol.x[5]; V_vals[3] = pf_sol.x[6]; V_vals[5] = pf_sol.x[7]
    P_res, Q_res = get_power_flow(th_vals, V_vals)
    return th_vals, V_vals, P_res, Q_res

def get_I_mat(V_vec, Y_lines):
    V_mat = np.zeros((len(V_vec), len(V_vec)), dtype=np.complex64)
    for idx, V in enumerate(V_vec):
        V_mat[idx] = -V_vec
        V_mat[idx] += V
        V_mat[idx, idx] = V
    return V_mat * Y_lines

In [ ]:
def objective_function(u):
    # Insert your code here
    # HINT 1: get_pf_sol expects pu values; u contains MW/pu voltages
    # HINT 2: Multiply the return value by S_base for numerical stability
    return

def opf_constraints(u):
    # Insert your code here
    # HINT: get_pf_sol expects pu values; u contains MW/pu voltages
    return

In [ ]:
# NOTE: Power constraints and bounds are in MW or Mvar.
#       Current constraints are in Ampere. Voltages are in pu.

const_min = []  # Insert lower bounds here
const_max = []  # Insert upper bounds here
bounds    = []  # Insert variable bounds here
const = NonlinearConstraint(opf_constraints, const_min, const_max)

In [ ]:
check_task_2_1(objective_function, opf_constraints)

### Task 6.2 — Run the power flow and OPF

Print the power flow solution **before** and **after** optimization.

**Expected result before optimization:**
| | Bus 1 | Bus 2 | Bus 3 | Bus 4 | Bus 5 | Bus 6 |
|--|-------|-------|-------|-------|-------|-------|
| P [MW] | −126.59 | −100.0 | 300.0 | −120.0 | 100.0 | −50.0 |
| Q [Mvar] | 90.47 | −50.0 | 35.67 | −60.0 | 18.72 | −25.0 |
| V [pu] | 1.02 | 1.0156 | 1.02 | 0.9985 | 1.02 | 1.0179 |
| δ [rad] | 0.0 | 0.0119 | 0.0335 | 0.0143 | 0.0141 | 0.0046 |

**Expected result after OPF:**
| | Bus 1 | Bus 2 | Bus 3 | Bus 4 | Bus 5 | Bus 6 |
|--|-------|-------|-------|-------|-------|-------|
| P [MW] | 76.20 | −100.0 | 154.16 | −120.0 | 41.46 | −50.0 |
| Q [Mvar] | 38.31 | −50.0 | 78.05 | −60.0 | 22.47 | −25.0 |
| V [pu] | 1.0493 | 1.0455 | 1.05 | 1.0284 | 1.0485 | 1.0469 |
| δ [rad] | 0.0 | −0.0037 | 0.0017 | −0.0131 | −0.0019 | −0.0032 |

In [ ]:
# Run the base power flow (before optimization) and print results


In [ ]:
# Run OPF and print results
sol_OPF = minimize(...)  # Insert arguments here

In [ ]:
check_task_2_2(sol_OPF)

---
### End of Task 6

## End of DigiLab Assignment
---